# YOLOv8 Fine-Tuning - Istanbul Trafigi
**ONEMLI:** Runtime > Change runtime type > T4 GPU secin!

In [ ]:
# Drive baglantisi (en basta yap ki egitim dogrudan Drive'a kaydedilsin)
import os
import shutil
from google.colab import drive

drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/tez_models'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Drive OK: {SAVE_DIR}")

In [ ]:
!pip install ultralytics roboflow -q

In [ ]:
import torch

print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("UYARI: GPU yok! Runtime > Change runtime type > T4 GPU secin!")

In [ ]:
# Roboflow'dan veri setini indir
from roboflow import Roboflow

rf = Roboflow(api_key="xT8j6ce4kyhvsVRwSaFr")
project = rf.workspace("sertas-workspace").project("istanbul-traffic-vehicles")
version = project.version(1)
dataset = version.download("yolov8")

DATASET_DIR = dataset.location
YAML_PATH = os.path.join(DATASET_DIR, "data.yaml")
print(f"Dataset: {DATASET_DIR}")
print(f"Icerik: {os.listdir(DATASET_DIR)}")

In [ ]:
# Train/Valid/Test bolme (eger sadece train varsa)
import random
import yaml

train_img = os.path.join(DATASET_DIR, 'train', 'images')
train_lbl = os.path.join(DATASET_DIR, 'train', 'labels')
valid_dir = os.path.join(DATASET_DIR, 'valid', 'images')

if not os.path.exists(valid_dir):
    print("valid klasoru yok, train'den bolunuyor...")
    images = sorted([f for f in os.listdir(train_img) if f.endswith(('.jpg', '.png'))])
    random.seed(42)
    random.shuffle(images)

    n = len(images)
    train_end = int(n * 0.7)
    val_end = int(n * 0.9)

    splits = {
        'valid': images[train_end:val_end],
        'test': images[val_end:]
    }

    for split_name, file_list in splits.items():
        img_dir = os.path.join(DATASET_DIR, split_name, 'images')
        lbl_dir = os.path.join(DATASET_DIR, split_name, 'labels')
        os.makedirs(img_dir, exist_ok=True)
        os.makedirs(lbl_dir, exist_ok=True)
        for img_file in file_list:
            lbl_file = os.path.splitext(img_file)[0] + '.txt'
            src_img = os.path.join(train_img, img_file)
            src_lbl = os.path.join(train_lbl, lbl_file)
            if os.path.exists(src_img):
                shutil.move(src_img, os.path.join(img_dir, img_file))
            if os.path.exists(src_lbl):
                shutil.move(src_lbl, os.path.join(lbl_dir, lbl_file))
        print(f"  {split_name}: {len(file_list)} gorsel")

    remaining = len(os.listdir(train_img))
    print(f"  train: {remaining} gorsel")
else:
    print("valid klasoru zaten var, bolme yapilmadi.")

# data.yaml guncelle
with open(YAML_PATH, 'r') as f:
    data = yaml.safe_load(f)

data['train'] = os.path.join(DATASET_DIR, 'train', 'images')
data['val'] = os.path.join(DATASET_DIR, 'valid', 'images')

with open(YAML_PATH, 'w') as f:
    yaml.dump(data, f)

print(f"\nSiniflar: {data['names']}")
print("data.yaml guncellendi.")

In [ ]:
# YOLOv8 Egitimi - DOGRUDAN DRIVE'A KAYDEDER
from ultralytics import YOLO

model = YOLO('yolov8s.pt')

results = model.train(
    data=YAML_PATH,
    epochs=50,
    imgsz=640,
    batch=16,
    name='istanbul_traffic_v1',
    patience=10,
    save=True,
    plots=True,
    project=SAVE_DIR,  # Dogrudan Drive'a kaydet
)

RESULTS_DIR = str(results.save_dir)
print(f"\nEgitim tamamlandi!")
print(f"Sonuclar (Drive'da): {RESULTS_DIR}")

In [ ]:
# Egitim grafikleri
from IPython.display import Image, display

for plot_name in ['results.png', 'confusion_matrix.png', 'val_batch0_pred.png']:
    plot_path = os.path.join(RESULTS_DIR, plot_name)
    if os.path.exists(plot_path):
        print(f"\n--- {plot_name} ---")
        display(Image(filename=plot_path, width=800))

In [ ]:
# Pretrained vs Fine-tuned karsilastirma
import pandas as pd

best_model_path = os.path.join(RESULTS_DIR, 'weights', 'best.pt')

pretrained = YOLO('yolov8s.pt')
finetuned = YOLO(best_model_path)

print("=== PRETRAINED (COCO) ===")
pretrained_metrics = pretrained.val(data=YAML_PATH)

print("\n=== FINE-TUNED (Istanbul) ===")
finetuned_metrics = finetuned.val(data=YAML_PATH)

comparison = pd.DataFrame({
    'Metrik': ['mAP@50', 'mAP@50-95', 'Precision', 'Recall'],
    'Pretrained': [
        f"{pretrained_metrics.box.map50:.4f}",
        f"{pretrained_metrics.box.map:.4f}",
        f"{pretrained_metrics.box.mp:.4f}",
        f"{pretrained_metrics.box.mr:.4f}",
    ],
    'Fine-tuned': [
        f"{finetuned_metrics.box.map50:.4f}",
        f"{finetuned_metrics.box.map:.4f}",
        f"{finetuned_metrics.box.mp:.4f}",
        f"{finetuned_metrics.box.mr:.4f}",
    ],
})

print("\n=== KARSILASTIRMA ===")
print(comparison.to_string(index=False))
print(f"\nModel Drive'da: {best_model_path}")